# 📄 Sistema de Extracción de Coordenadas PDF-JSON

## Funcionalidad del Proyecto

Este sistema extrae **bounding boxes (coordenadas)** de campos específicos en archivos PDF usando archivos JSON como referencia.

### Entrada:
- 📄 **PDF** con texto
- 📝 **JSON** con campos y valores a buscar

### Salida:
- 📊 **JSON con coordenadas** de cada campo encontrado
- Formato: `{"campo": "valor", "bbox": [x0, y0, x1, y1], "page": 0}`

### Uso:
- Crear datasets para entrenar modelos de Document Understanding (LayoutLM, etc.)
- Automatización de extracción de datos de documentos
- Análisis de documentos estructurados

## 1️⃣ Clonar Repositorio e Instalar Dependencias

In [ ]:
!git clone https://github.com/GynoRomeroPrado/Consultas-Claude.git
%cd Consultas-Claude

In [ ]:
# Instalar dependencias
!pip install -q PyMuPDF rapidfuzz numpy pandas tqdm Pillow

## 2️⃣ Verificar Archivos Disponibles

In [ ]:
!python verificar_archivos.py

In [ ]:
# Listar archivos PNG y JSON
!echo "=== Archivos PNG ==="
!ls "Datos extraidos de Originales/facturas_procesadas/" | head -5
!echo ""
!echo "=== Archivos JSON ==="
!ls "Datos extraidos de Originales/anotaciones/" | head -5

## 3️⃣ Ver Ejemplo de JSON

In [ ]:
import json
from pathlib import Path

# Leer primer JSON
json_files = list(Path("Datos extraidos de Originales/anotaciones").glob("*.json"))
primer_json = json_files[0]

print(f"📝 Archivo: {primer_json.name}\n")

with open(primer_json, 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f"Total de campos: {len(data)}\n")
print("Primeros 10 campos:")
print("=" * 80)

for i, (key, value) in enumerate(list(data.items())[:10], 1):
    print(f"{i:2}. {key:30} : {str(value)[:50]}")

print(f"\n... y {len(data) - 10} campos más")

## 4️⃣ Convertir PNG a PDF

In [ ]:
# Convertir todas las imágenes PNG a PDF
!python convertir_png_a_pdf_simple.py

In [ ]:
# Verificar PDFs creados
!ls "Datos extraidos de Originales/pdfs/" | wc -l
!echo "PDFs creados:"
!ls "Datos extraidos de Originales/pdfs/" | head -5

## 5️⃣ Ejecutar Prueba con UNA Factura

In [ ]:
# Probar con una sola factura primero
from pathlib import Path

# Obtener primer PDF y JSON
pdfs = sorted(list(Path("Datos extraidos de Originales/pdfs").glob("*.pdf")))
json_dir = Path("Datos extraidos de Originales/anotaciones")

if pdfs:
    pdf_path = pdfs[0]
    json_path = json_dir / (pdf_path.stem + ".json")
    
    print(f"PDF:  {pdf_path.name}")
    print(f"JSON: {json_path.name}")
    
    # Ejecutar extracción
    !python extract_coordinates.py "{pdf_path}" "{json_path}" -o output/test_primera_factura.json --fuzzy --threshold 75 -v
else:
    print("❌ No se encontraron PDFs")

## 6️⃣ Ver Resultados de la Prueba

In [ ]:
import json

# Leer resultado
with open('output/test_primera_factura.json', 'r', encoding='utf-8') as f:
    resultado = json.load(f)

print("📊 RESULTADOS\n" + "=" * 80)
print(f"Total de campos:     {resultado['total_fields']}")
print(f"Campos encontrados:  {len(resultado['annotations'])}")
print(f"Tasa de éxito:       {len(resultado['annotations']) / resultado['total_fields'] * 100:.1f}%")

print("\n📝 PRIMEROS 5 CAMPOS ENCONTRADOS")
print("=" * 80)

for i, ann in enumerate(resultado['annotations'][:5], 1):
    print(f"\n[{i}] {ann['field_name']}")
    print(f"    Texto: '{ann['text']}'")
    print(f"    Bbox:  ({ann['bbox']['x0']:.1f}, {ann['bbox']['y0']:.1f}, {ann['bbox']['x1']:.1f}, {ann['bbox']['y1']:.1f})")
    print(f"    Confianza: {ann['confidence']:.2f}")
    print(f"    Tipo: {ann['match_type']}")

## 7️⃣ Ejecutar Pruebas COMPLETAS (15 Facturas)

In [ ]:
# Ejecutar script completo de pruebas
!python test_facturas.py

## 8️⃣ Análisis de Resultados

In [ ]:
import json
from pathlib import Path
import pandas as pd

# Recolectar estadísticas de todos los resultados
results_dir = Path("output/facturas")
all_results = []

for result_file in results_dir.glob("*_auto.json"):
    with open(result_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    all_results.append({
        'archivo': result_file.stem,
        'total_campos': data['total_fields'],
        'encontrados': len(data['annotations']),
        'match_rate': len(data['annotations']) / data['total_fields'] * 100 if data['total_fields'] > 0 else 0
    })

if all_results:
    df = pd.DataFrame(all_results)
    
    print("📊 RESUMEN DE TODAS LAS FACTURAS")
    print("=" * 80)
    print(df.to_string(index=False))
    
    print("\n📈 ESTADÍSTICAS GENERALES")
    print("=" * 80)
    print(f"Match rate promedio:  {df['match_rate'].mean():.1f}%")
    print(f"Match rate mínimo:    {df['match_rate'].min():.1f}%")
    print(f"Match rate máximo:    {df['match_rate'].max():.1f}%")
    print(f"Total facturas:       {len(df)}")
else:
    print("No se encontraron resultados")

## 9️⃣ Exportar a Diferentes Formatos

In [ ]:
# Exportar a formato LayoutLM (para entrenamiento)
from pathlib import Path

pdfs = list(Path("Datos extraidos de Originales/pdfs").glob("*.pdf"))
if pdfs:
    primer_pdf = pdfs[0]
    json_path = Path("Datos extraidos de Originales/anotaciones") / (primer_pdf.stem + ".json")
    
    print("Exportando a formato LayoutLM...")
    !python extract_coordinates.py "{primer_pdf}" "{json_path}" -o output/layoutlm_example.json --format layoutlm
    
    print("\nExportando a formato COCO...")
    !python extract_coordinates.py "{primer_pdf}" "{json_path}" -o output/coco_example.json --format coco
    
    print("\nExportando a CSV...")
    !python extract_coordinates.py "{primer_pdf}" "{json_path}" -o output/example.csv --format csv
    
    print("\n✅ Exportaciones completadas")
    !ls -lh output/

## 🔟 Ver Ejemplo de Formato LayoutLM

In [ ]:
import json

with open('output/layoutlm_example.json', 'r', encoding='utf-8') as f:
    layoutlm_data = json.load(f)

print("📄 FORMATO LAYOUTLM (normalizado 0-1000)\n")
print(f"Formato: {layoutlm_data['format']}")
print(f"Escala: {layoutlm_data['scale']}")
print(f"Páginas: {len(layoutlm_data['pages'])}")

if layoutlm_data['pages']:
    primera_pagina = layoutlm_data['pages'][0]
    print(f"\nPágina 0:")
    print(f"  Ancho: {primera_pagina['width']}")
    print(f"  Alto: {primera_pagina['height']}")
    print(f"  Anotaciones: {len(primera_pagina['annotations'])}")
    
    print("\nPrimeras 3 anotaciones:")
    for i, ann in enumerate(primera_pagina['annotations'][:3], 1):
        print(f"\n  [{i}] {ann['label']}")
        print(f"      Texto: {ann['text']}")
        print(f"      Bbox normalizado: {ann['bbox']}")
        print(f"      Tipo: {ann['match_type']}")

## 📦 Descargar Resultados

In [ ]:
# Comprimir resultados
!zip -r resultados_facturas.zip output/

from google.colab import files
files.download('resultados_facturas.zip')

print("✅ Resultados descargados")